In [1]:
import json
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# 1. ĐỌC DỮ LIỆU & XÂY DỰNG TỪ ĐIỂN
# ==========================================
print("1. Đọc dữ liệu và xây dựng Từ điển (Vocab)...")
with open("/kaggle/input/datasets/xuantudo/codesearchnet-java/codesearchnet_java_50k.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Tách 80% Train, 10% Validation, 10% Test
train_data_raw = dataset[:40000]
val_data_raw = dataset[40000:45000]
test_data_raw = dataset[45000:]

MAX_CODE_LEN = 150
MAX_SUMMARY_LEN = 20

source_vocab_counter = Counter()
target_vocab_counter = Counter()

for item in train_data_raw:
    # Tách từ đơn giản bằng khoảng trắng (Text-based approach)
    code_words = item['code'].replace('{', ' { ').replace('}', ' } ').replace(';', ' ; ').split()[:MAX_CODE_LEN]
    sum_words = item['summary'].lower().replace('.', ' .').split()[:MAX_SUMMARY_LEN]
    
    source_vocab_counter.update(code_words)
    target_vocab_counter.update(sum_words)

# Khai báo token đặc biệt
SPECIAL_TOKENS = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}

def build_vocab(counter, min_freq=3):
    vocab = SPECIAL_TOKENS.copy()
    idx = 4
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

source_vocab = build_vocab(source_vocab_counter, min_freq=5)
target_vocab = build_vocab(target_vocab_counter, min_freq=3)
target_vocab_inv = {v: k for k, v in target_vocab.items()}

print(f"-> Source Vocab (Code): {len(source_vocab)} từ")
print(f"-> Target Vocab (Summary): {len(target_vocab)} từ")

1. Đọc dữ liệu và xây dựng Từ điển (Vocab)...
-> Source Vocab (Code): 21958 từ
-> Target Vocab (Summary): 7377 từ


In [2]:
# ==========================================
# 2. CHUYỂN TEXT THÀNH TENSORS
# ==========================================
print("\n2. Chuyển đổi Dữ liệu thành Tensor...")

def process_data(raw_data):
    processed = []
    
    # Kích thước cố định cho Summary (Cộng 2 vì chứa thêm <SOS> và <EOS>)
    FIXED_TARGET_LEN = MAX_SUMMARY_LEN + 2 
    
    for item in raw_data:
        # 1. Mã hóa Code (Giữ nguyên)
        code_words = item['code'].replace('{', ' { ').replace('}', ' } ').replace(';', ' ; ').split()[:MAX_CODE_LEN]
        code_ids = [source_vocab.get(w, 3) for w in code_words]
        code_ids += [0] * (MAX_CODE_LEN - len(code_ids)) # Padding cho Code
        code_tensor = torch.tensor([code_ids], dtype=torch.long)
        
        # 2. Mã hóa Summary (THÊM PADDING VÀO ĐÂY)
        sum_words = item['summary'].lower().replace('.', ' .').split()[:MAX_SUMMARY_LEN]
        sum_ids = [1] + [target_vocab.get(w, 3) for w in sum_words] + [2]
        
        # Lấp đầy số 0 (<PAD>) nếu câu summary ngắn hơn FIXED_TARGET_LEN
        sum_ids += [0] * (FIXED_TARGET_LEN - len(sum_ids))
        
        target_tensor = torch.tensor(sum_ids, dtype=torch.long)
        
        processed.append((code_tensor, target_tensor))
        
    return processed

train_dataset = process_data(train_data_raw)
val_dataset = process_data(val_data_raw)
test_dataset = process_data(test_data_raw)
print(f"-> Đã tạo xong: Tập Train ({len(train_dataset)}) | Tập Val ({len(val_dataset)}) | Tập Test ({len(test_dataset)})")


2. Chuyển đổi Dữ liệu thành Tensor...
-> Đã tạo xong: Tập Train (40000) | Tập Val (5000) | Tập Test (4783)


In [3]:
# ==========================================
# 3. KIẾN TRÚC MÔ HÌNH BASELINE
# ==========================================
print("\n3. Khởi tạo Mô hình Baseline (Seq2Seq)...")
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(TextEncoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        return hidden

class GRUDecoder(nn.Module):
    def __init__(self, target_vocab_size, embedding_dim, hidden_dim):
        super(GRUDecoder, self).__init__()
        self.embedding = nn.Embedding(target_vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, target_vocab_size)

    def forward(self, input_word, hidden_state):
        embedded = self.embedding(input_word) 
        output, new_hidden_state = self.gru(embedded, hidden_state)
        prediction = self.fc(output.squeeze(1)) 
        return prediction, new_hidden_state

EMBED_DIM = 128
HIDDEN_DIM = 128

encoder = TextEncoder(vocab_size=len(source_vocab), embedding_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM)
decoder = GRUDecoder(target_vocab_size=len(target_vocab), embedding_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM)

encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("-> Mô hình đã sẵn sàng!")


3. Khởi tạo Mô hình Baseline (Seq2Seq)...
-> Mô hình đã sẵn sàng!


In [4]:
from torch.utils.data import DataLoader, TensorDataset

# ==========================================
# 4. VÒNG LẶP HUẤN LUYỆN (TỐI ƯU HÓA BATCHING)
# ==========================================
print("\n4. Chuẩn bị Dữ liệu theo Lô (Batching)...")

# Ép danh sách các tensor thành 1 Tensor lớn để chia Batch
def create_dataloader(dataset, batch_size):
    # dataset là list các tuple: (code_tensor [1, seq_len], target_tensor [seq_len])
    codes = torch.cat([item[0] for item in dataset], dim=0) 
    # Bọc thêm 1 chiều cho target để stack
    targets = torch.stack([item[1] for item in dataset], dim=0) 
    
    tensor_dataset = TensorDataset(codes, targets)
    return DataLoader(tensor_dataset, batch_size=batch_size, shuffle=True)

BATCH_SIZE = 128 # Mỗi lần huấn luyện 128 mẫu cùng lúc
train_loader = create_dataloader(train_dataset, BATCH_SIZE)
val_loader = create_dataloader(val_dataset, BATCH_SIZE)

print("Bắt đầu Huấn luyện siêu tốc...")

def train_step_batched(code_batch, target_batch, encoder, decoder, enc_opt, dec_opt, criterion):
    enc_opt.zero_grad()
    dec_opt.zero_grad()
    loss = 0
    valid_steps = 0 # Đếm số lượng bước giải mã không bị nan

    hidden = encoder(code_batch)
    current_batch_size = code_batch.size(0)
    decoder_input = torch.tensor([[1]] * current_batch_size, dtype=torch.long)
    target_length = target_batch.size(1)

    for t in range(target_length):
        output, hidden = decoder(decoder_input, hidden)
        step_loss = criterion(output, target_batch[:, t])

        # Bức tường phòng thủ: Chỉ cộng nếu loss không phải là nan
        if not torch.isnan(step_loss):
            loss += step_loss
            valid_steps += 1

        decoder_input = target_batch[:, t].unsqueeze(1)

    # Đảm bảo chỉ backward nếu có tensor hợp lệ (có từ thực tế được sinh ra)
    if isinstance(loss, torch.Tensor):
        loss.backward()
        enc_opt.step()
        dec_opt.step()
        return loss.item() / valid_steps
        
    return 0.0

def evaluate_loss_batched(val_loader, encoder, decoder, criterion):
    encoder.eval()
    decoder.eval()
    total_loss = 0

    with torch.no_grad():
        for code_batch, target_batch in val_loader:
            hidden = encoder(code_batch)
            current_batch_size = code_batch.size(0)
            decoder_input = torch.tensor([[1]] * current_batch_size, dtype=torch.long)
            target_length = target_batch.size(1)
            loss = 0
            valid_steps = 0

            for t in range(target_length):
                output, hidden = decoder(decoder_input, hidden)
                step_loss = criterion(output, target_batch[:, t])

                # Bức tường phòng thủ
                if not torch.isnan(step_loss):
                    loss += step_loss
                    valid_steps += 1

                decoder_input = target_batch[:, t].unsqueeze(1)

            if isinstance(loss, torch.Tensor):
                total_loss += loss.item() / valid_steps

    return total_loss / len(val_loader)

EPOCHS = 10
best_val_loss = float('inf')
patience = 2 
trigger_times = 0

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    epoch_loss = 0
    
    print(f"\n[Epoch {epoch+1}/{EPOCHS}] Đang huấn luyện...")
    for batch_idx, (code_batch, target_batch) in enumerate(train_loader):
        step_loss = train_step_batched(code_batch, target_batch, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        epoch_loss += step_loss
        
        # In tiến độ sau mỗi 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f"   Đã xử lý {(batch_idx + 1) * BATCH_SIZE}/{len(train_dataset)} mẫu... Loss: {step_loss:.4f}")
            
    train_loss_avg = epoch_loss / len(train_loader)
    
    print("   Đang tính điểm Validation...")
    val_loss_avg = evaluate_loss_batched(val_loader, encoder, decoder, criterion)
    
    print(f"=== KẾT QUẢ EPOCH {epoch+1} ===")
    print(f" - Train Loss: {train_loss_avg:.4f}")
    print(f" - Val Loss:   {val_loss_avg:.4f}")
    
    if val_loss_avg < best_val_loss:
        best_val_loss = val_loss_avg
        trigger_times = 0
        torch.save(encoder.state_dict(), 'seq2seq_encoder_best.pth')
        torch.save(decoder.state_dict(), 'seq2seq_decoder_best.pth')
        print(" -> MÔ HÌNH TỐT LÊN! Đã lưu lại trọng số.")
    else:
        trigger_times += 1
        print(f" -> Cảnh báo: Val Loss không giảm ({trigger_times}/{patience})")
        if trigger_times >= patience:
            print("=> EARLY STOPPING: Đã dừng huấn luyện để chống Overfitting!")
            break


4. Chuẩn bị Dữ liệu theo Lô (Batching)...
Bắt đầu Huấn luyện siêu tốc...

[Epoch 1/10] Đang huấn luyện...
   Đã xử lý 6400/40000 mẫu... Loss: 4.8650
   Đã xử lý 12800/40000 mẫu... Loss: 4.6403
   Đã xử lý 19200/40000 mẫu... Loss: 4.3622
   Đã xử lý 25600/40000 mẫu... Loss: 4.0936
   Đã xử lý 32000/40000 mẫu... Loss: 4.2052
   Đã xử lý 38400/40000 mẫu... Loss: 3.8464
   Đang tính điểm Validation...
=== KẾT QUẢ EPOCH 1 ===
 - Train Loss: 4.6556
 - Val Loss:   4.2161
 -> MÔ HÌNH TỐT LÊN! Đã lưu lại trọng số.

[Epoch 2/10] Đang huấn luyện...
   Đã xử lý 6400/40000 mẫu... Loss: 3.6352
   Đã xử lý 12800/40000 mẫu... Loss: 3.6225
   Đã xử lý 19200/40000 mẫu... Loss: 3.9199
   Đã xử lý 25600/40000 mẫu... Loss: 3.4391
   Đã xử lý 32000/40000 mẫu... Loss: 3.1215
   Đã xử lý 38400/40000 mẫu... Loss: 3.4778
   Đang tính điểm Validation...
=== KẾT QUẢ EPOCH 2 ===
 - Train Loss: 3.6774
 - Val Loss:   3.9858
 -> MÔ HÌNH TỐT LÊN! Đã lưu lại trọng số.

[Epoch 3/10] Đang huấn luyện...
   Đã xử lý 6400/

In [5]:
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import numpy as np
import torch
import torch.nn.functional as F 

nltk.download('wordnet')
nltk.download('omw-1.4')

# ... (Giữ nguyên các hàm beam_search và evaluate_baseline bên dưới) ...

# 1. Hàm Beam Search dành riêng cho Seq2Seq (Baseline)
def beam_search_decode_text(code_tensor, encoder, decoder, target_vocab_inv, beam_width=3, max_length=20):
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        # Lấy ngữ cảnh từ TextEncoder thay vì GGNN
        initial_hidden = encoder(code_tensor)
        
    beams = [(0.0, [1], initial_hidden)] # 1 là <SOS>
    
    for _ in range(max_length):
        new_beams = []
        for score, seq, hidden in beams:
            if seq[-1] == 2: # 2 là <EOS>
                new_beams.append((score, seq, hidden))
                continue
                
            last_token = torch.tensor([[seq[-1]]], dtype=torch.long)
            with torch.no_grad():
                output, new_hidden = decoder(last_token, hidden)
                
            log_probs = F.log_softmax(output.squeeze(0), dim=-1)
            topk_log_probs, topk_indices = log_probs.topk(beam_width)
            
            for i in range(beam_width):
                word_idx = topk_indices[i].item()
                word_score = topk_log_probs[i].item()
                new_beams.append((score + word_score, seq + [word_idx], new_hidden))
                
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_width]
        
    best_seq = beams[0][1]
    words = [target_vocab_inv.get(idx, "<UNK>") for idx in best_seq if idx not in [0, 1, 2]]
    return " ".join(words)

# 2. Hàm Đánh giá và in điểm
def evaluate_baseline(test_dataset, encoder, decoder, target_vocab_inv, target_vocab):
    print(f"Đang chấm điểm trên {len(test_dataset)} mẫu Test...")
    references = []
    hypotheses = []
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l_scores, meteor_scores = [], []

    # Tải trọng số tốt nhất đã lưu ở Epoch 2
    encoder.load_state_dict(torch.load('seq2seq_encoder_best.pth'))
    decoder.load_state_dict(torch.load('seq2seq_decoder_best.pth'))

    for idx, (code_tensor, target_tensor) in enumerate(test_dataset):
        # Đáp án đúng
        ref_ids = target_tensor.tolist()
        ref_words = [target_vocab_inv.get(w, "<UNK>") for w in ref_ids if w not in [0, 1, 2]]
        ref_sentence = " ".join(ref_words)
        
        # Dự đoán
        pred_sentence = beam_search_decode_text(code_tensor, encoder, decoder, target_vocab_inv, beam_width=3)
        pred_words = pred_sentence.split()
        
        references.append([ref_words])
        hypotheses.append(pred_words)
        
        if len(pred_words) > 0 and len(ref_words) > 0:
            rouge_l_scores.append(scorer.score(ref_sentence, pred_sentence)['rougeL'].fmeasure)
            meteor_scores.append(meteor_score([ref_words], pred_words))
            
        if (idx + 1) % 500 == 0:
            print(f"  Đã dịch {idx + 1}/{len(test_dataset)} mẫu...")

    smooth = SmoothingFunction().method4
    bleu_4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    
    print("\n" + "="*40)
    print(" 🥉 KẾT QUẢ BASELINE (SEQ2SEQ) 🥉")
    print("="*40)
    print(f" - BLEU-4 Score:  {bleu_4 * 100:.2f}")
    print(f" - METEOR Score:  {np.mean(meteor_scores) * 100:.2f}")
    print(f" - ROUGE-L Score: {np.mean(rouge_l_scores) * 100:.2f}")
    print("="*40)

# GỌI HÀM ĐỂ LẤY ĐIỂM
evaluate_baseline(test_dataset, encoder, decoder, target_vocab_inv, target_vocab)

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Đang chấm điểm trên 4783 mẫu Test...
  Đã dịch 500/4783 mẫu...
  Đã dịch 1000/4783 mẫu...
  Đã dịch 1500/4783 mẫu...
  Đã dịch 2000/4783 mẫu...
  Đã dịch 2500/4783 mẫu...
  Đã dịch 3000/4783 mẫu...
  Đã dịch 3500/4783 mẫu...
  Đã dịch 4000/4783 mẫu...
  Đã dịch 4500/4783 mẫu...

 🥉 KẾT QUẢ BASELINE (SEQ2SEQ) 🥉
 - BLEU-4 Score:  1.10
 - METEOR Score:  15.48
 - ROUGE-L Score: 15.15
